# CO_covid_response_lag — COVID-19 Response Lag of Rolling Conformal Threshold

Per-model response lag of the rolling conformal threshold q̂_{V,t} to the
COVID-19 volatility shock on S&P 500. Response lag is measured from the
realised-volatility peak (2020-03-27) to the date when q̂_{V,t} first
reaches its post-shock maximum.

**Outputs:** `covid_response_lags.csv`, `fig_covid_response.pdf`/`.png`

In [ ]:
"""
COVID-19 response lag of rolling conformal threshold.
Measured from realised-vol peak to first q_V maximum.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SCRIPT_DIR = Path('.').resolve()
BASE = SCRIPT_DIR.parent
DATA = BASE / 'cfp_ijf_data' / 'paper_outputs' / 'tables'
FIG_DIR = BASE / 'figures'
OUT = SCRIPT_DIR

RVOL_PEAK = pd.Timestamp('2020-03-27')
SHOCK_START = pd.Timestamp('2020-02-20')
SHOCK_END = pd.Timestamp('2020-04-30')
PLOT_START = '2019-07-01'
PLOT_END = '2021-07-31'

TSFM_MODELS = ['Chronos-Small', 'Chronos-Mini', 'TimesFM-2.5',
               'Moirai-2.0', 'Lag-Llama']
PARAM_MODELS = ['GJR-GARCH', 'GARCH-N', 'Hist-Sim', 'EWMA']
ALL_MODELS = TSFM_MODELS + PARAM_MODELS


def compute_lags(df):
    rows = []
    for model in ALL_MODELS:
        series = df[model].dropna()
        post = series[(series.index >= RVOL_PEAK) &
                      (series.index <= '2021-01-01')]
        if len(post) == 0:
            continue
        post_max = post.max()
        first_max_date = post[post == post_max].index[0]
        lag_cal = (first_max_date - RVOL_PEAK).days
        lag_biz = len(df.loc[RVOL_PEAK:first_max_date]) - 1

        pre = series[(series.index >= '2019-11-01') &
                     (series.index < SHOCK_START)]
        pre_level = pre.mean() if len(pre) > 0 else np.nan

        shock_window = series[(series.index >= SHOCK_START) &
                              (series.index <= SHOCK_END)]
        shock_max = shock_window.max() if len(shock_window) > 0 else np.nan
        shock_max_date = (shock_window.idxmax().strftime('%Y-%m-%d')
                          if len(shock_window) > 0 else '')
        ratio = shock_max / pre_level if pre_level != 0 else np.nan

        group = 'TSFM' if model in TSFM_MODELS else 'Parametric'
        rows.append({
            'model': model, 'group': group,
            'pre_shock_mean': pre_level,
            'shock_window_max': shock_max,
            'shock_max_date': shock_max_date,
            'shock_ratio': ratio,
            'post_shock_max': post_max,
            'first_max_date': first_max_date.strftime('%Y-%m-%d'),
            'lag_calendar_days': lag_cal,
            'lag_business_days': lag_biz,
        })
    return pd.DataFrame(rows)


def make_figure(df):
    fig, ax = plt.subplots(figsize=(14, 5.5))
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    plot_df = df.loc[PLOT_START:PLOT_END]

    tsfm_colors = {
        'Chronos-Small': '#CD0000',
        'Chronos-Mini':  '#E8AFAF',
        'TimesFM-2.5':   '#3F51B5',
        'Moirai-2.0':    '#2196F3',
        'Lag-Llama':     '#795548',
    }
    param_colors = {
        'GARCH-N':    '#BDBDBD',
        'GJR-GARCH':  '#9E9E9E',
        'Hist-Sim':   '#A0A0A0',
        'EWMA':       '#C0C0C0',
    }

    for model in TSFM_MODELS:
        s = plot_df[model].dropna()
        if len(s) > 0:
            ax.plot(s.index, s.values, lw=1.5,
                    color=tsfm_colors[model], label=model)

    for model in PARAM_MODELS:
        s = plot_df[model].dropna()
        if len(s) > 0:
            ax.plot(s.index, s.values, lw=1.0, alpha=0.7,
                    color=param_colors[model], label=model)

    ax2 = ax.twinx()
    rvol = plot_df['rvol'].dropna()
    ax2.fill_between(rvol.index, 0, rvol.values,
                     color='#CCCCCC', alpha=0.35)
    ax2.set_ylabel('Realised vol.', fontsize=11)
    ax2.set_ylim(0, 1.05)

    ax.axvspan(SHOCK_START, SHOCK_END,
               alpha=0.12, color='#CD0000', zorder=0)
    ax.annotate('COVID-19\nshock',
                xy=(pd.Timestamp('2020-03-15'), ax.get_ylim()[1] * 0.92),
                fontsize=9, color='#CD0000', ha='center', style='italic')

    ax.set_ylabel(r'$\hat{q}_{V,t}$', fontsize=12)
    ax.set_xlabel('Date', fontsize=11)
    ax.legend(loc='upper left', fontsize=8, framealpha=0.9, ncol=3)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(axis='y', alpha=0.2)

    plt.tight_layout()
    for ext in ['pdf', 'png']:
        fig.savefig(OUT / f'fig_covid_response.{ext}',
                    dpi=300, bbox_inches='tight', transparent=True)
    FIG_DIR.mkdir(exist_ok=True)
    for ext in ['pdf', 'png']:
        fig.savefig(FIG_DIR / f'fig_covid_response.{ext}',
                    dpi=300, bbox_inches='tight', transparent=True)
    plt.show()
    plt.close(fig)


# ── Main ──────────────────────────────────────────────────────
print(f'Reference date (rvol peak): {RVOL_PEAK.date()}')

df = pd.read_csv(DATA / 'rolling_qv_SP500.csv',
                 index_col=0, parse_dates=True)

lags = compute_lags(df)
lags.to_csv(OUT / 'covid_response_lags.csv', index=False)

print('Per-model response lags (calendar days from rvol peak):')
print(lags.to_string(index=False))

tsfm_mean = lags[lags['group'] == 'TSFM']['lag_calendar_days'].mean()
param_mean = lags[lags['group'] == 'Parametric']['lag_calendar_days'].mean()
print(f'\nTSFM mean lag:       {tsfm_mean:.0f} calendar days')
print(f'Parametric mean lag: {param_mean:.0f} calendar days')

make_figure(df)
print('Saved: fig_covid_response.pdf/.png, covid_response_lags.csv')